1. Ma’lumotlarni tayyorlash





Kerakli kutubxonalarni import qiling.



torchvision.datasets orqali FashionMNIST ma’lumotlar to‘plamini yuklang (train va test uchun alohida).



Ma’lumotlarni ToTensor va Normalize((0.5,), (0.5,)) orqali o‘zgartiring.



DataLoader yarating. batch_size qiymatini 32 qilib belgilang.



2. CNN Modelini qurish





Videodarsdagi kabi nn.Module’dan meros oluvchi yangi KiyimCNN klassini yarating.



nn.Sequential yordamida model arxitekturasini quyidagi parametarlar bilan quring: 1-blok: Conv2d (chiqish kanallari: 8), ReLU, MaxPool2d .



Diqqat: Flatten qatlamidan keyingi birinchi Linear qatlamining kirish neyronlari sonini (in_features) o‘zingiz hisoblab toping.



Modelni yarating va uni device’ga o‘tkazing.



3. Modelni shug‘ullantirish va baholash





Yo‘qotish funksiyasi (criterion) sifatida nn.CrossEntropyLoss'ni tanlang.



Optimizator (optimizer) sifatida optim.Adam’dan foydalaning, lr (o‘rganish darajasi) qiymatini 0.002 qiling.



Modelni 5 epoxa davomida shug‘ullantiring. Har bir epoxadan so‘ng o‘rtacha yo‘qotishni (total_loss) ekranga chiqaring.



Shug‘ullantirish tugagach, test ma’lumotlari yordamida model aniqligini (accuracy) hisoblang va chop eting.



In [14]:
# PyTorch asosiy kutubxonalari
import torch
import torch.nn as nn
import torch.optim as optim

# Dataset va transformlar uchun
from torchvision import datasets, transforms

# DataLoader
from torch.utils.data import DataLoader

In [15]:
# Agar GPU mavjud bo‘lsa ishlatadi, aks holda CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [16]:
# Transform: rasmni tensorga o'tkazish va normalizatsiya qilish
transform = transforms.Compose([
    transforms.ToTensor(),                 # 0-255 -> 0-1
    transforms.Normalize((0.5,), (0.5,))   # -1 dan +1 oralig'iga keltiradi
])

# Train dataset yuklash
train_data = datasets.FashionMNIST(
    root="./data",     # saqlanadigan papka
    train=True,        # train dataset
    download=True,     # agar yo'q bo'lsa yuklaydi
    transform=transform
)

# Test dataset yuklash
test_data = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# DataLoader yaratish
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

CNN

In [17]:
# CNN model klassi
class KiyimCNN(nn.Module):
    def __init__(self):
        super(KiyimCNN, self).__init__()

        # Feature extraction qismi (Conv blok)
        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,     # grayscale image (1 kanal)
                out_channels=8,    # 8 ta filter
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),            # aktivatsiya funksiyasi
            nn.MaxPool2d(2)       # 28x28 -> 14x14
        )

        # Flatten + Fully connected qatlamlar
        self.classifier = nn.Sequential(
            nn.Flatten(),

            # HISOB:
            # 28x28 -> Conv -> 28x28
            # MaxPool -> 14x14
            # Channel = 8
            # 8 * 14 * 14 = 1568
            nn.Linear(8 * 14 * 14, 128),

            nn.ReLU(),
            nn.Linear(128, 10)    # 10 ta class (FashionMNIST)
        )

    def forward(self, x):
        x = self.features(x)      # convolution qismi
        x = self.classifier(x)    # classification qismi
        return x


# Modelni yaratish va device ga o'tkazish
model = KiyimCNN().to(device)

print(model)

KiyimCNN(
  (features): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1568, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [18]:
# Loss function (classification uchun)
criterion = nn.CrossEntropyLoss()

# Optimizer (Adam)
optimizer = optim.Adam(
    model.parameters(),
    lr=0.002   # learning rate
)

In [19]:
epochs = 5

for epoch in range(epochs):
    model.train()   # train mode
    total_loss = 0

    for images, labels in train_loader:
        # Data ni device ga o'tkazish
        images = images.to(device)
        labels = labels.to(device)

        # Gradientlarni tozalash
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Loss hisoblash
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Weightlarni update qilish
        optimizer.step()

        total_loss += loss.item()

    # Har epoch uchun o'rtacha loss
    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/5], Loss: 0.4170
Epoch [2/5], Loss: 0.2879
Epoch [3/5], Loss: 0.2464
Epoch [4/5], Loss: 0.2159
Epoch [5/5], Loss: 0.1918


In [20]:
model.eval()   # evaluation mode

correct = 0
total = 0

# Gradient hisoblanmaydi (tezroq ishlaydi)
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        # Eng katta ehtimollikni olish
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Accuracy hisoblash
accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 89.97%


Bu mashqda biz PyTorch yordamida FashionMNIST datasetida ishlaydigan oddiy CNN (Convolutional Neural Network) modelini yaratdik.

Avvalo, ma’lumotlarni yuklab, ToTensor va Normalize orqali tayyorladik hamda DataLoader bilan batchlarga ajratdik.
So‘ng nn.Module asosida KiyimCNN modelini qurib, unda Conv2d → ReLU → MaxPool → Flatten → Linear qatlamlaridan foydalandik.
in_features qiymatini rasm o‘lchamlariga asoslanib (8 × 14 × 14 = 1568) hisoblab topdik.
Modelni CrossEntropyLoss va Adam optimizer yordamida 5 epoxa davomida o‘qitdik.
Yakunda test dataset orqali model aniqligini (accuracy) baholadik.